# Architecture

```mermaid
flowchart LR
  R[Python repository] --> P[Parser Service]
  P --> N[cpg.nodes.v1]
  P --> E[cpg.edges.v1]
  P --> M[cpg.source-metadata.v1]
  P --> X[cpg.parser-errors.v1]
  N --> C[Kafka Connect]
  E --> C
  C --> G[Neo4j]
  C --> Q[cpg.neo4j-dlq.v1]
  M --> S[Spark Structured Streaming]
  S --> D[MongoDB]
  S --> K[(Checkpoint)]
  X --> L[Parser error evidence]
```

Graph topology goes directly from Kafka Connect to Neo4j. Spark is used only
for source metadata; parser failures and connector failures have separate paths.

## Why the pipeline is split

The graph and metadata paths solve different problems. Neo4j needs individual
nodes and relationships, whereas MongoDB needs one current document for each
source file. Keeping those contracts separate at Kafka lets the Neo4j connector
consume topology directly and leaves Spark responsible only for metadata and
its checkpoint. It also means that a parser failure or a bad connector record
has its own visible error path instead of blocking valid traffic.

We considered sending every event through one Spark job. That would reduce the
number of arrows in the diagram, but it would also couple two unrelated schemas
and make Spark an unnecessary relay for Neo4j, contrary to the assignment. The
single broker, one partition per topic, and replication factor one are deliberate
demo choices: they make ordering and replay easier to inspect, but they are not
presented as a production deployment.

In [1]:
import subprocess
from pathlib import Path

root = Path('..').resolve()
result = subprocess.run(
    ['docker', 'compose', 'config', '--services'],
    cwd=root, capture_output=True, text=True, check=True,
)
print(result.stdout.rstrip())
required = {'broker', 'connect', 'neo4j', 'mongo', 'spark-metadata'}
assert required <= set(result.stdout.splitlines())
print('PASS: all required architecture services are declared')

broker
kafka-init
mongo
neo4j
neo4j-init
spark-metadata
connect
connect-init
PASS: all required architecture services are declared


## Reflection

The final run confirmed that topology reaches Neo4j without
passing through Spark, while metadata advances independently through the Spark
checkpoint. The first connector startup exposed an integration race: Kafka
Connect accepted the registration request before its task was ready, so an
immediate status check sometimes returned 404. `register-wait.sh` now retries
until both the connector and task report `RUNNING`. The remaining limitation is
intentional and visible in the diagram: this is a single-node teaching stack,
not a highly available Kafka deployment.